# MediaPipe Landmark Extraction to CSV

This notebook uses MediaPipe Holistic to extract face (468), pose (33), and both hands (21 each) landmarks frame-by-frame from a video and saves them to a CSV with columns similar to `landmarks [1].csv`.

Notes:
- Coordinates are MediaPipe normalized image coordinates (x,y in [0,1], z in a roughly meter-based canonical depth scale).
- Hand mapping: `hand_0_*` = left hand, `hand_1_*` = right hand.
- Missing landmarks for a frame/component are stored as blank (NaN in CSV).
- Configure the input video and output path below.
- Orientation: set ROTATE_MODE to keep frames horizontal automatically (e.g., 'auto_landscape'), or force 'cw'/'ccw'/'180'/'none'.

In [1]:
# If running for the first time in this environment, ensure dependencies are installed
# You can skip if already installed via requirements.txt
import sys, subprocess, pkgutil

def ensure(package, pip_name=None):
    pip_name = pip_name or package
    if pkgutil.find_loader(package) is None:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

# mediapipe now uses the 'mediapipe' package name (0.10+)
ensure('mediapipe')
ensure('opencv-python', 'opencv-python')
ensure('pandas')
ensure('numpy')

Installing opencv-python...


In [2]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from pathlib import Path

# Config
VIDEO_PATH = Path('palmas.mp4')  # change if needed
OUTPUT_CSV = Path('landmarks_output.csv')
DATASET_NAME = 'custom'  # free-form label
EXTRACTOR_NAME = 'mediapipe-holistic'

# Orientation control
# ROTATE_MODE: 'none' | 'cw' | 'ccw' | '180' | 'auto_landscape'
# - 'auto_landscape': if frame is taller than wide (portrait), rotate 90 CW to landscape
ROTATE_MODE = 'auto_landscape'

# MediaPipe setup
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

# Landmark sizes (fixed by MediaPipe models)
N_FACE = 468
N_POSE = 33
N_HAND = 21  # each hand

# Build CSV header similar to your sample
static_cols = [
    'category', 'video_name', 'frame', 'person', 'video_path', 'dataset', 'extractor'
]

# hands: left (0) and right (1)
hand_cols = []
for h in [0, 1]:
    for i in range(N_HAND):
        hand_cols += [f'hand_{h}_{i}_x', f'hand_{h}_{i}_y', f'hand_{h}_{i}_z']

# face 0..467
face_cols = []
for i in range(N_FACE):
    face_cols += [f'face_{i}_x', f'face_{i}_y', f'face_{i}_z']

# pose 0..32
pose_cols = []
for i in range(N_POSE):
    pose_cols += [f'pose_{i}_x', f'pose_{i}_y', f'pose_{i}_z']

COLUMNS = static_cols + hand_cols + face_cols + pose_cols

COLUMNS[:10], len(COLUMNS)

(['category',
  'video_name',
  'frame',
  'person',
  'video_path',
  'dataset',
  'extractor',
  'hand_0_0_x',
  'hand_0_0_y',
  'hand_0_0_z'],
 1636)

In [3]:
def to_row(video_path: Path, frame_idx: int, results, video_name: str, category: str = '', person: int = 0):
    """Convert a holistic results object into one CSV row (list) matching COLUMNS order.
    Missing parts are filled with np.nan.
    Hands mapping: 0=left, 1=right.
    """
    row = [
        category,
        video_name,
        frame_idx,
        person,
        str(video_path),
        DATASET_NAME,
        EXTRACTOR_NAME,
    ]

    # Helper to flatten landmark list of size N into [x,y,z]*N, np.nan if missing
    def flatten_landmarks(landmark_list, expected_n):
        if landmark_list is None or landmark_list.landmark is None:
            return [np.nan] * (expected_n * 3)
        lm = landmark_list.landmark
        if len(lm) != expected_n:
            # pad or cut if size mismatch (shouldn't happen in normal runs)
            vals = []
            for i in range(expected_n):
                if i < len(lm):
                    vals += [lm[i].x, lm[i].y, lm[i].z]
                else:
                    vals += [np.nan, np.nan, np.nan]
            return vals
        vals = []
        for i in range(expected_n):
            vals += [lm[i].x, lm[i].y, lm[i].z]
        return vals

    # Hands
    left_hand = results.left_hand_landmarks if results else None
    right_hand = results.right_hand_landmarks if results else None
    row += flatten_landmarks(left_hand, N_HAND)
    row += flatten_landmarks(right_hand, N_HAND)

    # Face
    face = results.face_landmarks if results else None
    row += flatten_landmarks(face, N_FACE)

    # Pose
    pose = results.pose_landmarks if results else None
    row += flatten_landmarks(pose, N_POSE)

    return row

# Quick check that row length matches columns size when nothing detected
_test_row = to_row(VIDEO_PATH, 0, None, VIDEO_PATH.name)
len(_test_row), len(COLUMNS)

(1636, 1636)

In [4]:
# Main extraction loop

def _rotate_frame(frame, mode: str):
    if mode == 'none' or mode is None:
        return frame
    if mode == 'cw':
        return cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
    if mode == 'ccw':
        return cv2.rotate(frame, cv2.ROTATE_90_COUNTERCLOCKWISE)
    if mode == '180':
        return cv2.rotate(frame, cv2.ROTATE_180)
    if mode == 'auto_landscape':
        h, w = frame.shape[:2]
        if h > w:
            return cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
        return frame
    return frame


def extract_video_to_csv(video_path: Path, output_csv: Path, category: str = '', person: int = 0,
                          max_frames: int | None = None,
                          detection_confidence: float = 0.5,
                          tracking_confidence: float = 0.5,
                          model_complexity: int = 1,
                          face: bool = True, pose: bool = True, hands: bool = True,
                          rotate_mode: str = None):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")

    # Determine which submodels to enable
    enable_face = face
    enable_pose = pose
    enable_hands = hands

    rows = []
    frame_idx = 0

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=model_complexity,
        smooth_landmarks=True,
        enable_segmentation=False,
        refine_face_landmarks=False,
        min_detection_confidence=detection_confidence,
        min_tracking_confidence=tracking_confidence,
    ) as holistic:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # Rotate if needed
            rm = rotate_mode if rotate_mode is not None else ROTATE_MODE
            frame = _rotate_frame(frame, rm)

            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(image_rgb)

            # If any of face/pose/hands is disabled, null them
            if not enable_face:
                results.face_landmarks = None
            if not enable_pose:
                results.pose_landmarks = None
            if not enable_hands:
                results.left_hand_landmarks = None
                results.right_hand_landmarks = None

            row = to_row(video_path, frame_idx, results, video_path.name, category=category, person=person)
            rows.append(row)

            frame_idx += 1
            if max_frames is not None and frame_idx >= max_frames:
                break

    cap.release()

    # Save DataFrame
    df = pd.DataFrame(rows, columns=COLUMNS)
    df.to_csv(output_csv, index=False)
    return df

# Run a short smoke test on a few frames to validate header/row size
smoke_df = extract_video_to_csv(VIDEO_PATH, OUTPUT_CSV.with_name('landmarks_smoke.csv'), max_frames=3,
                                rotate_mode=ROTATE_MODE)
len(smoke_df), smoke_df.shape

I0000 00:00:1760837392.811784 7142850 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1760837392.890599 7142971 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1760837392.903392 7142978 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1760837392.905677 7142979 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1760837392.905688 7142971 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1760837392.905719 7142973 inference_feedback_manager.cc:114] Feedback manager requires a

(3, (3, 1636))

In [5]:
# Full run (set max_frames=None to process all frames)
FULL_OUTPUT_CSV = Path('landmarks_full.csv')
# df = extract_video_to_csv(VIDEO_PATH, FULL_OUTPUT_CSV, category='', person=0, max_frames=None, rotate_mode=ROTATE_MODE)
# display(df.head())
print(f"To process all frames: df = extract_video_to_csv({VIDEO_PATH}, Path('{FULL_OUTPUT_CSV.name}'), rotate_mode='{ROTATE_MODE}')")

To process all frames: df = extract_video_to_csv(palmas.mp4, Path('landmarks_full.csv'), rotate_mode='auto_landscape')


In [6]:
df = extract_video_to_csv(VIDEO_PATH, FULL_OUTPUT_CSV, rotate_mode=ROTATE_MODE)
len(df), df.shape

I0000 00:00:1760837393.083825 7142850 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
W0000 00:00:1760837393.152535 7143001 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1760837393.160569 7143006 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1760837393.161785 7143003 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1760837393.161798 7143008 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1760837393.162148 7143000 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling supp

(194, (194, 1636))

In [1]:
import cv2
cap = cv2.VideoCapture("17RuimSinalizador06-2.mp4")
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
print("frames:", n_frames, "fps:", fps)


frames: 126 fps: 30.0


In [3]:
import cv2
import os

video_path = "17RuimSinalizador06-2.mp4"
exists = os.path.exists(video_path)

info = {"file_exists": exists, "video_path": video_path}
if exists:
    cap = cv2.VideoCapture(video_path)
    frame_count_prop = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    # compute duration if fps > 0
    duration_sec = frame_count_prop / fps if fps and fps > 0 else None
    cap.release()
    info.update({
        "frame_count": frame_count_prop,
        "fps": fps,
        "resolution": f"{width}x{height}",
        "duration_sec": duration_sec
    })

info


{'file_exists': True,
 'video_path': '17RuimSinalizador06-2.mp4',
 'frame_count': 126,
 'fps': 30.0,
 'resolution': '1920x1080',
 'duration_sec': 4.2}